<a href="https://colab.research.google.com/github/SSolanoRuniandes/Notebooks-Aprendizaje-por-Refuerzo-Profundo/blob/main/TareaSemana3_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![MAIA banner](https://raw.githubusercontent.com/MAIA4361-Aprendizaje-refuerzo-profundo/Notebooks_Tareas/main/Images/Aprendizaje_refuerzo_profundo_Banner_V1.png)

# <h1><center>Tarea Tutorial - Semana 3 <a href="https://colab.research.google.com/github/SSolanoRuniandes/Notebooks-Aprendizaje-por-Refuerzo-Profundo/blob/main/TareaSemana3_v2.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

<center><h1>Deep Q Networks</h1></center>

En este notebook se presenta una introducción práctica en detalle al algoritmo de DQN y sus variantes, como Doble DQN y QRDQN. Para ello se va a abordar un problema que se ha convertido en un estándar para la prueba de algoritmos de aprendizaje por refuerzo: jugar videojuegos de Atari. En este tutorial aprenderás sobre cómo se pueden utilizar redes neuronales convolucionales para entrenar algoritmos directamente a partir de imágenes, y sobre algunas técnicas o estrategias que se han desarrollado para mejorar el aprendizaje. Nos concentraremos específicamente en el juego <i>Freeway</i>, encontrado dentro de las librería Gymnasium, y las implementaciones de DQN ofrecidas por Stable-Baselines3.


# Tabla de Contenidos
1. [Objetivos de Aprendizaje](#scrollTo=Objetivos_de_Aprendizaje)  
2. [Marco Teórico](#scrollTo=Marco_Te_rico)  
3. [Instalación de Librerías](#scrollTo=Instalaci_n_de_Librer_as)  
4. [Familiarización con el Entorno de Gym](#scrollTo=Familiarizaci_n_con_el_Entorno_de_Gym)  
5. [DQN](#scrollTo=DQN)
6. [DQN con pasos muertos?](#scrollTo=DQN_pasos_muertos)
7. [QR-DQN](#scrollTo=QR_DQN)  
8. [Prioritized Experience Replay?](#scrollTo=Prioritized_Experience_Replay)
9. [Dueling Networks?](#scrollTo=Dueling_Networks)  
10. [Reflexiones Finales](#scrollTo=Reflexiones_Finales)  
11. [Referencias](#scrollTo=Referencias)

# Objetivos de Aprendizaje  

A través de los ejercicios de este tutorial se busca:
  

*   Evidenciar el uso de redes neuronales convolucionales para aprender directamente de imágenes en un entorno/problema de Aprendizaje por Refuerzo.
*   Comprender las bases teóricas del funcionamiento del algoritmo de Deep Q-Networks (DQN).
*   Experimental con algunas modificaciones útiles que se aplican al algoritmo de DQN, como Doble DQN, Quantile Regression DQN (QR-DQN) o Prioritized Experience Replay, y observar sus ventajas.




# Marco Teórico  

Anteriormente, se abordó el método tabular de Q-Learning, un algoritmo de aprendizaje off-policy. En el año 2013, Minh et al. [1] publicaron un artículo donde modificaban este algoritmo para utilizar aproximación de funciones haciendo uso de redes neuronales convolucionales. Este paper introdujo así el algoritmo de Deep Q-Networks (DQN), y lo usó de forma exitosa en juegos de Atari. El algoritmo de DQN se muestra en la Figura 1 a continuación.

![DQNdF](https://raw.githubusercontent.com/MAIA4361-Aprendizaje-refuerzo-profundo/Notebooks_Tareas/main/Images/DQN_dF.png)


<center>Figura 1. Algoritmo de DQN.</center>


Minh et al. utilizaron este algoritmo usando como entradas de la red neuronal convolucional imágenes de 84x84 pixeles en escala de grises, correspondientes a la secuencia de distintos juegos de Atari. La salida de la red era el valor de la función Q para cada una de las posibles acciones que puede tomar el jugador con los controles disponibles. Este paper marcó un antes y un después en el Aprendizaje por Refuerzo profundo, porque en este caso se está aprendiendo directamente a partir de las imágenes, mientras que anteriormente se empleaban representaciones lineales más sencillas o modelos diseñados específicamente para cada problema. En este caso, una misma red neuronal, modelo y parámetros se pueden utilizar para distintos juegos sin tener conocimiento previo de los mismos. Adicionalmente, en este artículo se introducen estrategias importantes como el uso de minibatches y Experience Replay. El Experience Replay consiste en guardar las experiencias pasadas del agente y usarlas posteriormente de diferentes partes del entrenamiento, rompiendo la correlación temporal de los datos. De esta manera, Minh et al. logran obtener resultados mejores en distintos juegos de Atari a comparación de métodos pasados y también logran superar en algunos casos a jugadores humanos expertos. [1]

No obstante, el algoritmo de DQN luego pasó por una serie de mejoras y cambios sugeridos en otras publicaciones. El avance más importante probablemente fue la implementación de Doble Doble DQN, propuesto en el artículo de Hasselt et al. [2] del año 2016. En Q-Learning tradicional y también en DQN, el algoritmo tiende a sobreestimar los valores de las acciones debido al sesgo de maximización, que aparece al tomar el máximo sobre acciones en la actualización de la función Q. Esta sobreestimación puede llevar a encontrar peores políticas. Para corregir este problema, en el artículo se generaliza el Doble Q-Learning (versión tabular) al algoritmo de DQN. Para ello recurren a dos redes neuronales: una red online para seleccionar las acciones, y otra red para realizar la evaluación, denominada target network. Con esto se consigue reducir la sobreestimación y mejorar la estabilidad del entrenamiento, resultando generalmente en mejores puntajes en los mismos juegos de Atari.

Otra modificación importante consiste en utilizar un Experience Replay priorizado (Prioritized Experience Replay). Esta estrategia fue propuesta por Schaul et al. [3] en 2016. Básicamente, en el Experience Replay original el agente guarda trancisiones como experiencia y las guarda al azar para romper las correlaciones temporales, pero no todas las experiencias son igual de útiles, y la repetición de trancisiones poco informativas termina realentizando el aprendizaje. Con Prioritized Experience Repaly, se utiliza la magnitud del error TD como medida de importancia, priorizando las trancisiones con mayor error TD, reproduciendo más a menudo estas experiencias para aprender más rápido. Esta priorización puede hacerse de forma proporcional al error TD o basada en la posición en un ranking.

El algoritmo de Doble DQN, con Prioritized Experience Replay proporcional, se muestra en la Figura 2.

![DobleDQNdF](https://raw.githubusercontent.com/MAIA4361-Aprendizaje-refuerzo-profundo/Notebooks_Tareas/main/Images/DobleDQN_dF.png)

<center>Figura 2. Algoritmo de Doble DQN con Prioritized Experience Replay proporcional.</center>


Finalmente, en 2017 Dabney et al. [4] proponen otra técnica llamada Quantile Regression DQN (QR-DQN). En este algoritmo no se predice únicamente el valor esperado de la recompensa, sino que se estima la distribución completa de posibles recompensas que el agente puede recibir. En general esta técnica puede llegar a ser más robusta y estable, ya que captura mejor la variabilidad e incertidumbre del entorno, pero también puede requerir mayores tiempos de entrenamiento.



# Instalación de Librerías  

Para este tutorial, siguiendo la metodología de los papers anteriormente mencionados, se va a recurrir a un juego de Atari 2600: <i>Freeway</i>. Este videojuego ya se encuentra incluido en los ambientes de Atari de la librería Gymnasium. También se utilizarán las implementaciones de DQN encontradas dentro de Stable-Baselines3.

Antes de comenzar, se sugiere elegir un entorno de simulación acelerado por GPU. En el caso de Colab gratuito, debería elegir el entorno de T4. Para ello diríjase a:

`Entorno de Ejecución > Cambiar Tipo de Entorno de Ejecución > GPU T4`


![DobleDQNdF](https://raw.githubusercontent.com/MAIA4361-Aprendizaje-refuerzo-profundo/Notebooks_Tareas/main/Images/t4.png)


Después, ejecute el siguiente bloque de código para instalar todas las librerías y herramientas necesarias.


In [ ]:
#Descarga librerías no incluidas en Colab usando pip
!pip install stable_baselines3 #Stable Baselines3 -> Framework de Reinforcement Learning
!pip install sb3-contrib #SB3-Contrib es un repositorio aparte con otros algoritmos
!pip install ale-py #ALE se utiliza para el ambiente de Atari
!pip install "gymnasium[atari,accept-rom-license]" stable-baselines3 autorom renderlab -q #Gymnasium, envs de Atari y ROM
!AutoROM --accept-license
!pip install renderlab #usado para renderizar gym

#Importa estas librerías
import stable_baselines3 #importa Stable Baselines3
from stable_baselines3 import DQN #importa el agente/algoritmo de DQN
from stable_baselines3.common.logger import configure #importa herramientas de logger/debug
from stable_baselines3.common.logger import Logger, CSVOutputFormat, HumanOutputFormat #importa herramientas de logger/debug
from stable_baselines3.common.evaluation import evaluate_policy #importa herramienta de evaluación automática
from sb3_contrib import QRDQN #importa el agente/algoritmo de QRDQN
import gymnasium #importa la libreria de gymnasium con las simulaciones
import renderlab #importa renderlab para los videos

import ale_py #importa ale para los ambientes de Atari
from gymnasium.wrappers import TimeLimit #importa timelimit para acortar los episodios
from stable_baselines3.common.env_util import make_atari_env #importa make_atari_env para escala de grises
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack #importa VecFrameStack para apilar frames y acelerar así el entrenamiento
from collections import deque #importa para ajustar los videos con VecFrameStack
import cv2 #importa para ajustar los videos con VecFrameStack

#!Importante:
gymnasium.register_envs(ale_py) #Hay que registrar los entornos de ALE manualmente!!!

#Importa otras librerías básicas
import numpy as np
import matplotlib.pyplot as plt
import random
import math
import pandas as pd
import sys

#Limpia los registros generados
from IPython.display import clear_output
clear_output()
print("Todas las librerías han sido instaladas correctamente.")

Todas las librerías han sido instaladas correctamente.


# Familiarización con el Entorno de Gym

En el videojuego <i>Freeway</i> [5], el jugador controla a un pollo cuyo objetivo es cruzar una calle por donde constantemente pasan automóviles de un lado a otro. Cada vez que el pollo logra llegar al otro lado de la acera, obtiene un punto y regresa a la posición inicial. Una imágen del juego se muestra en la Figura 3.


![freeway](https://raw.githubusercontent.com/MAIA4361-Aprendizaje-refuerzo-profundo/Notebooks_Tareas/main/Images/freeway.png)

<center>Figura 3. Ejemplo de una partida de Freeway.</center>

En este ambiente el jugador tiene 3 posibles acciones en cada paso de tiempo:


*   0 - Se queda quieto (NOOP).
*   1 - Se mueve hacia arriba (UP).
*   2 - Se mueve hacia abajo (DOWN).

Adicionalmente, para configurar una partida de Freeway hay dos parámetros importantes: modo (mode) y dificultad (difficulty). El modo regula la cantidad de autos que se mueven en la carretera, con un número entre 0 y 7. En modo 0 hay menos autos, mientras que en el modo 7 hay más autos y son más rápidos. Por otro lado, la dificultad puede ser 0 o 1. En dificultad 0, cuando el pollo es atropellado este no regresa al inicio de la carretera, sino que sólo regresa una línea hacia atrás. Al contrario, cuando la dificultad es 1, cuando el pollo es atropellado el jugador debe reiniciar desde el inicio de la carretera.

Dado el tiempo de juego, en general un jugador humano experto puede esperar obtener una puntuación entre 20 y 25.





## Ejemplo

A continuación se muestra un ejemplo de simulación de una partida de <i>Freeway</i>. En este caso, se usa el modo 0 y dificultad 0 (ambiente más fácil posible), y siempre se dirige el pollo hacia arriba (acción en 1).

In [ ]:
#Parámetros del ambiente
mode=0
difficulty=0

env_render = gymnasium.make("ALE/Freeway-v5", render_mode="rgb_array", mode=mode, difficulty=difficulty) #Se crea el ambiente.
env_render = renderlab.RenderFrame(env_render, "./output") #Se crea una copia que se pueda renderizar con renderlab

terminated = False #Inicializa una condición para el loop
truncated = False #Inicializa una condición para el loop
total_reward=0 #Inicializa contador del retorno

obs , info = env_render.reset() #Se reinicia el estado para comenzar. En obs se almacena el estado observado (continuo, 2 dimensiones)

while not (terminated or truncated): #Simula hasta que termine la partida
  action = 1 #Siempre va hacia arriba
  obs, reward, terminated, truncated, info = env_render.step(action) #da el paso
  total_reward+=reward #acumula recompensa

print("Recompensa obtenida en el episodio:",total_reward) #Se imprime la recompensa obtenida
print("\n\n")

env_render.play() #Con esta función se obtiene el video de la simulación

Recompensa obtenida en el episodio: 21.0



Moviepy - Building video temp-{start}.mp4.
Moviepy - Writing video temp-{start}.mp4



Moviepy - Done !
Moviepy - video ready temp-{start}.mp4


## Ejercicio Práctico

En el anterior ejemplo, se puede evidenciar que el agente obtiene una puntuación muy buena (entre 20 y 25). En este caso, ir siempre hacia adelante resultó ser una buena política porque con dificultad 0 el pollito no regresa al inicio de la carretera, por lo cual termina siendo poco relevante si el pollito intenta en su lugar esquivar a los automóviles, ya que el efecto sería similar.

¿Si cambiamos la dificultad a 1, y cambiamos el modo a 7, esta estrategia resulta ser igual de efectiva?

In [ ]:
# Ejecute una partida con dificultad 1 y modo 7

# =====================================================
# COMPLETAR ===========================================
#

# =====================================================

In [ ]:
#Parámetros del ambiente
mode=7
difficulty=1

env_render = gymnasium.make("ALE/Freeway-v5", render_mode="rgb_array", mode=mode, difficulty=difficulty) #Se crea el ambiente.
env_render = renderlab.RenderFrame(env_render, "./output") #Se crea una copia que se pueda renderizar con renderlab

terminated = False #Inicializa una condición para el loop
truncated = False #Inicializa una condición para el loop
total_reward=0 #Inicializa contador del retorno

obs , info = env_render.reset() #Se reinicia el estado para comenzar. En obs se almacena el estado observado (continuo, 2 dimensiones)

while not (terminated or truncated): #Simula hasta que termine la partida
  action = 1 #Siempre va hacia arriba
  obs, reward, terminated, truncated, info = env_render.step(action) #da el paso
  total_reward+=reward #acumula recompensa

print("Recompensa obtenida en el episodio:",total_reward) #Se imprime la recompensa obtenida
print("\n\n")

env_render.play() #Con esta función se obtiene el video de la simulación

Recompensa obtenida en el episodio: 5.0



Moviepy - Building video temp-{start}.mp4.
Moviepy - Writing video temp-{start}.mp4



Moviepy - Done !
Moviepy - video ready temp-{start}.mp4


En este caso, el puntaje fue mucho más bajo (cercano a 5). Así se puede ver que ir hacia adelante siempre no resulta ser una estrategia viable para obtener un alto puntaje cuando la dificultad y el modo aumentan. El objetivo del entrenamiento será entonces encontrar una política que aprenda a jugar de forma óptima en estas condiciones, o por lo menos una política que obtenga mejores resultados.

#DQN

En primer lugar, para intentar resolver este problema, utilizaremos una versión <i>vanilla</i> de DQN, es decir, sin doble DQN ni Prioritized Experience Replay o similar, como hizo Mnih et al en 2013 [1]. Esta es la versión que se encuentra implementada dentro de Stable-Baselines3.

## Ejemplo

Iniciaremos nuevamente con el caso más sencillo posible: modo 0 y dificultad 0. A continuación, se muestra un ejemplo de cómo debe crearse el agente de DQN con una serie de parámetros optimizado, el proceso de entrenamiento y posterior render de la política. Tenga en cuenta algunos aspectos importantes:

*   Se hace uso de la función <i>make_atari_env</i> de SB3. Esta función automáticamente prepara el ambiente de Gymansium para hacer más ligero y eficiente el entrenamiento. Principalmente se encarga de convertir las imágenes del juego en escala de grises y también reescalarlas. Adicionalmente, se puede utilizar también para entrenar varios ambientes en paralelo, pero en este caso se usará únicamente un ambiente.
*   Se hace uso de la función <i>VecFrameStack</i> de SB3. Cuando se usa una única imágen como entrada de la red neuronal, no se tiene información sobre velocidad o dirección de movimiento, por lo cual conviene usar esta función para apilar varios frames y tener mayor información en el entrenamiento. Esto permite realizar entrenamientos en tiempos más cortos, lo cual es importante si se trabaja en una sesión de colab. En este caso, se apilarán 4 frames.
*   Como se apilan 4 frames, las dimensiones del modelo cambian con respecto a las de la familiarización anterior. Por ende, se requieren hacer unos ajustes manuales para que el video sea un continuo y no se vea recortado cada 4 frames.
*   Como la entrada son imágenes, se debe trabajar con una política <i>CnnPolicy</i> (Convolutional Neural Networks).
*   El tamaño de buffer se ve limitado por la sesión de Colab. Por defecto está en 100,000, y la RAM no es suficiente para esto con una sesión gratuita. Para evitar errores, se recomienda reducirlo a 50,000.
*   Se recomienda usar el entorno de ejecución acelerado por GPU T4 (disponible en la versión gratuita).
*   Se muestran unos logs de entrenamiento cada 5 epidosios y después el video.
*   El modelo entrenado se guarda temporalmente en la sesión de Colab como un archivo .zip, el cual puede cargar en otras celdas. También puede descargarlo y volverlo a subir en otra sesión para no tener que volver a entrenar un modelo.  

`Tiempo aproximado que demora el entrenamiento: 10 minutos`


In [ ]:
#Parámetros del ejemplo
modo=0
dificultad=0

env = make_atari_env(
    "ALE/Freeway-v5",
    n_envs=1,
    seed=0,
    env_kwargs={"mode": modo, "difficulty": dificultad}
) #se utilizará sólo un ambiente

env = VecFrameStack(env, n_stack=4)  # apila 4 frames para dar percepción del movimiento

#Útil: Limitar el tiempo del episodio
#env = TimeLimit(env, max_episode_steps=512)

#Importante: Limitar el buffer por las limitaciones de Colab
#Útil: learning_Starts para ganar algo de experiencia (2 episodios)
model = DQN(
    "CnnPolicy",
    env,
    learning_rate=0.0001,
    buffer_size=50_000,         #original: 100_000
    learning_starts=1024,       #original: 100
    batch_size=32,
    gamma=0.99,
    train_freq=4,
    target_update_interval=10000,
    exploration_fraction=0.7,   #original: 0.1
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,  #original: 0.05
    policy_kwargs=None,
    verbose=1)

#100 episodios de entrenamiento
model.learn(total_timesteps=51_200, log_interval=5)
model.save("dqn_freeway_1")



# Cargar el modelo entrenado
model = DQN.load("dqn_freeway_1")

# Crea un entorno para renderizar
env = gymnasium.make("ALE/Freeway-v5", render_mode="rgb_array", mode=modo, difficulty=dificultad)
env = renderlab.RenderFrame(env, "./output")

# Frame stack manual para el modelo (gris 84x84)
frame_stack = deque(maxlen=4)

# Reset
obs, info = env.reset()

# Procesa una copia del frame sólo para el modelo
def preprocess(obs):
    gray = obs.mean(axis=2).astype(np.uint8)  # escala de grises
    resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return resized

# Llena el frame stack inicial
preprocessed = preprocess(obs)
for _ in range(4):
    frame_stack.append(preprocessed)

terminated = False
truncated = False
total_reward = 0

# Loop de simulación
while not (terminated or truncated):
    stacked_obs = np.stack(frame_stack, axis=0)  # (4, 84, 84) aquí se ajustan los frames
    action, _ = model.predict(stacked_obs, deterministic=True)

    # Avanza en el entorno real manteniendo la continuidad en el video
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    # Actualiza el stack sólo para el modelo
    preprocessed = preprocess(obs)
    frame_stack.append(preprocessed)

print("Recompensa obtenida en el episodio:", total_reward)
env.play()

Using cuda device
Wrapping the env in a VecTransposeImage.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0.6      |
|    exploration_rate | 0.933    |
| time/               |          |
|    episodes         | 5        |
|    fps              | 164      |
|    time_elapsed     | 15       |
|    total_timesteps  | 2545     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 9e-08    |
|    n_updates        | 380      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0.8      |
|    exploration_rate | 0.865    |
| time/               |          |
|    episodes         | 10       |
|    fps              | 161      |
|    time_elapsed     | 31       |
|    total_timesteps  | 5092     |
| train/              |          |
|    learning_rate    | 0.0001 

Moviepy - Done !
Moviepy - video ready temp-{start}.mp4


# Ejercicio Práctico

Se puede observar en el ejemplo anterior que el pollito termina yendo siempre hacia adelante, obteniendo una recompensa igual o muy similar a la obtenida anteriormente en el proceso de familiarización. Esto no significa que la política sea mala, pero en definitiva el algoritmo de DQN no es muy útil en este caso. Evaluemos mejor su desempeño con otros parámetros. Repita los entrenamientos cambiando los parámetros:

*   modo 0 y dificultad 1.
*   modo 7 y dificultad 1.

También puede ajustar algunos de los siguientes parámetros del modelo y entrenamiento para intentar obtener mejores resultados:

*   <i>learning_rate</i>: La tasa de aprendizaje.
*   <i>batch_size</i>: Cantidad de experiencias almacenadas en el Experience Replay.
*   <i>exploration_fraction</i>: Porcentaje del entrenamiento en el cual el agente reduce progresivamente su tasa de exploración ($\epsilon$).
*   <i>exploration_final_eps</i>: La tasa de exploración ($\epsilon$) que mantendrá el agente al final.
*   <i>total_timesteps</i>: Equivalente al tiempo de entrenamiento.

In [ ]:
modo=0
dificultad=1

env = make_atari_env(
    "ALE/Freeway-v5",
    n_envs=1,
    seed=0,
    env_kwargs={"mode": modo, "difficulty": dificultad}
)

env = VecFrameStack(env, n_stack=4)  # apila 4 frames para dar percepción del movimiento

#Útil: Limitar el tiempo del episodio
#env = TimeLimit(env, max_episode_steps=512)

#Importante: Limitar el buffer por las limitaciones de Colab
#Útil: learning_Starts para ganar algo de experiencia (2 episodios)
model = DQN(
    "CnnPolicy",
    env,
    learning_rate=0.0001,
    buffer_size=50_000,         #original: 100_000
    learning_starts=1024,       #original: 100
    batch_size=32,
    gamma=0.99,
    train_freq=4,
    target_update_interval=10000,
    exploration_fraction=0.7,   #original: 0.1
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,  #original: 0.05
    policy_kwargs=None,
    verbose=1)

#100 episodios de entrenamiento
model.learn(total_timesteps=51_200, log_interval=5)
model.save("dqn_freeway_2")


# Cargar el modelo entrenado
model = DQN.load("dqn_freeway_2")

# Crea un entorno para renderizar
env = gymnasium.make("ALE/Freeway-v5", render_mode="rgb_array", mode=modo, difficulty=dificultad)
env = renderlab.RenderFrame(env, "./output")

# Frame stack manual para el modelo (apilamos en gris 84x84)
frame_stack = deque(maxlen=4)

# Reset del entorno
obs, info = env.reset()

# Procesa una copia del frame sólo para el modelo
def preprocess(obs):
    gray = obs.mean(axis=2).astype(np.uint8)  # escala de grises
    resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return resized

# Llena el frame stack inicial
preprocessed = preprocess(obs)
for _ in range(4):
    frame_stack.append(preprocessed)

terminated = False
truncated = False
total_reward = 0

# Loop de simulación
while not (terminated or truncated):
    stacked_obs = np.stack(frame_stack, axis=0)  # (4, 84, 84) aquí se ajustan los frames
    action, _ = model.predict(stacked_obs, deterministic=True)

    # Avanza en el entorno real manteniendo la continuidad en el video
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    # Actualiza el stack sólo para el modelo
    preprocessed = preprocess(obs)
    frame_stack.append(preprocessed)

print("Recompensa obtenida en el episodio:", total_reward)
env.play()

Using cuda device
Wrapping the env in a VecTransposeImage.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.933    |
| time/               |          |
|    episodes         | 5        |
|    fps              | 135      |
|    time_elapsed     | 18       |
|    total_timesteps  | 2545     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 5.48e-07 |
|    n_updates        | 380      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.865    |
| time/               |          |
|    episodes         | 10       |
|    fps              | 142      |
|    time_elapsed     | 35       |
|    total_timesteps  | 5092     |
| train/              |          |
|    learning_rate    | 0.0001 

Moviepy - Done !
Moviepy - video ready temp-{start}.mp4


In [ ]:
modo=7
dificultad=1

env = make_atari_env(
    "ALE/Freeway-v5",
    n_envs=1,
    seed=0,
    env_kwargs={"mode": modo, "difficulty": dificultad}
)

env = VecFrameStack(env, n_stack=4)  # apila 4 frames para dar percepción del movimiento

#Útil: Limitar el tiempo del episodio
#env = TimeLimit(env, max_episode_steps=512)

#Importante: Limitar el buffer por las limitaciones de Colab
#Útil: learning_Starts para ganar algo de experiencia (2 episodios)
model = DQN(
    "CnnPolicy",
    env,
    learning_rate=0.0001,
    buffer_size=50_000,         #original: 100_000
    learning_starts=1024,       #original: 100
    batch_size=32,
    gamma=0.99,
    train_freq=4,
    target_update_interval=10000,
    exploration_fraction=0.7,   #original: 0.1
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,  #original: 0.05
    policy_kwargs=None,
    verbose=1)

#100 episodios de entrenamiento
model.learn(total_timesteps=51_200, log_interval=5)
model.save("dqn_freeway_3")

# Cargar el modelo entrenado
model = DQN.load("dqn_freeway_3")

# Crea un entorno para renderizar
env = gymnasium.make("ALE/Freeway-v5", render_mode="rgb_array", mode=modo, difficulty=dificultad)
env = renderlab.RenderFrame(env, "./output")

# Frame stack manual para el modelo (apilamos en gris 84x84)
frame_stack = deque(maxlen=4)

# Reset del entorno
obs, info = env.reset()

# Procesa una copia del frame sólo para el modelo
def preprocess(obs):
    gray = obs.mean(axis=2).astype(np.uint8)  # escala de grises
    resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return resized

# Llena el frame stack inicial
preprocessed = preprocess(obs)
for _ in range(4):
    frame_stack.append(preprocessed)

terminated = False
truncated = False
total_reward = 0

# Loop de simulación
while not (terminated or truncated):
    stacked_obs = np.stack(frame_stack, axis=0)  # (4, 84, 84) aquí se ajustan los frames
    action, _ = model.predict(stacked_obs, deterministic=True)

    # Avanza en el entorno real manteniendo la continuidad en el video
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    # Actualiza el stack sólo para el modelo
    preprocessed = preprocess(obs)
    frame_stack.append(preprocessed)

print("Recompensa obtenida en el episodio:", total_reward)
env.play()

Using cuda device
Wrapping the env in a VecTransposeImage.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.933    |
| time/               |          |
|    episodes         | 5        |
|    fps              | 172      |
|    time_elapsed     | 14       |
|    total_timesteps  | 2545     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 4.84e-07 |
|    n_updates        | 380      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0.1      |
|    exploration_rate | 0.865    |
| time/               |          |
|    episodes         | 10       |
|    fps              | 163      |
|    time_elapsed     | 31       |
|    total_timesteps  | 5092     |
| train/              |          |
|    learning_rate    | 0.0001 

  warnings.warn(



Recompensa obtenida en el episodio: 5.0
Moviepy - Building video temp-{start}.mp4.
Moviepy - Writing video temp-{start}.mp4



Moviepy - Done !
Moviepy - video ready temp-{start}.mp4


Ahora sí podemos observar que el algoritmo de DQN continúa aprendiendo simplemente en ir en línea recta, obteniendo un mal puntaje como se había observado anteriormente en la familiarización. Esto depende en gran medida del tiempo de enetrenamiento disponible y la exploración realizada, pero nos interesa realizar cambios en el algoritmo para lograr obtener mejores resultados en tiempos similares. En las secciones a continuación se implementan algunas de las herramientas vistas anteriormente en el marco teórico y se comparan los resultados obtenidos en dichos casos con los de DQN <i>vanilla</i>.

# **DQN - Pasos muertos
Hay pasos muertos al principio.

Sin nada, ni doble ni prrioritized experience replay

In [ ]:
from gymnasium import Wrapper

class DelayActionWrapper(Wrapper):
    def __init__(self, env, delay_steps=100):
        super().__init__(env)
        self.delay_steps = delay_steps
        self.current_step = 0

    def reset(self, **kwargs):
        self.current_step = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        if self.current_step < self.delay_steps:
            action = 0
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.current_step += 1
        return obs, reward, terminated, truncated, info

In [ ]:
from gymnasium.wrappers import TimeLimit

gymnasium.register_envs(ale_py) #Hay que registrar los entornos de ALE manualmente
env = gymnasium.make("ALE/Freeway-v5", mode=0, difficulty=0) #Crea el ambiente de Freeway

#Útil: Limitar el tiempo del episodio
env = DelayActionWrapper(env, delay_steps=512)

#Importante: Limitar el buffer por las limitaciones de Colab
#Útil: learning_Starts para ganar algo de experiencia (4 episodios)
model = DQN("CnnPolicy", env, buffer_size=50_000, verbose=1, learning_starts=4096)

#100 episodios de entrenamiento
model.learn(total_timesteps=204_800, log_interval=5)
model.save("dqn_freeway")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.525    |
| time/               |          |
|    episodes         | 5        |
|    fps              | 260      |
|    time_elapsed     | 39       |
|    total_timesteps  | 10240    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 2.11e-07 |
|    n_updates        | 1535     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 10       |
|    fps              | 192      |
|    time_elapsed     | 106      |
|    total_timesteps  | 

In [ ]:
model = DQN.load("dqn_freeway")


env_render = gymnasium.make("ALE/Freeway-v5", render_mode="rgb_array", mode=0, difficulty=0) #Se crea el ambiente. Para este tutorial, utilice gymnasium si va a renderizar.
#env_render = DelayActionWrapper(env, delay_steps=512, no_op_action=0)
env_render = renderlab.RenderFrame(env_render, "./output") #Se crea una copia que se pueda renderizar con renderlab

terminated = False #Inicializa una condición para el loop
truncated = False #Inicializa una condición para el loop
total_reward=0 #Inicializa contador del retorno

obs , info = env_render.reset() #Se reinicia el estado para comenzar. En obs se almacena el estado observado (continuo, 2 dimensiones)

step=0
while not (terminated or truncated): #Simula hasta que el carro salga del valle o hasta que pasen 200 episodios
  action, _states = model.predict(obs, deterministic=True)
  #if step < 512:
  #      action = 0

  obs, reward, terminated, truncated, info = env_render.step(action)
  total_reward+=reward
  step += 1

print("Recompensa obtenida en el episodio:",total_reward) #Se imprime la recompensa obtenida
print("\n\n")

env_render.play() #Con esta función se obtiene el video de la simulación


Recompensa obtenida en el episodio: 0.0



Moviepy - Building video temp-{start}.mp4.
Moviepy - Writing video temp-{start}.mp4



Moviepy - Done !
Moviepy - video ready temp-{start}.mp4


# **QR-DQN**

Ahora procederemos a usar una de las variantes más recientes de DQN: Quantile Regression DQN. Este algoritmo se encuentra en Stable-Baselines3 Contrib (que existe como una unidad aparte de SB3, y continene una serie de algoritmos un poco más experimentales).

Al igual que otras extensiones de Deep Q-Learning (como Doble DQN), QR-DQN surge para solucionar las deficiencias del DQN clásico. Conserva la estructura de dos redes (red principal y red objetivo) para mantener un aprendizaje estable, pero reemplaza la estimación escalar del valor $Q$ por el modelado de la distribución del retorno, permitiendo tomar decisiones más precisas en entornos complejos.

Adicionalmente, en este caso se duplican los pasos de entrenamiento, y se vuelve a suministrar un modelo con parámetros optimizados. Tenga en cuenta las observaciones y recomendaciones dadas anteriormente.

## Ejemplo
Omitiendo el caso de modo 0 y dificultad 0, que comienza a ser trivial, probaremos ahora el caso de modo 0 y dificultad 1.

`Tiempo aproximado que demora el entrenamiento: 20 minutos`

In [ ]:
modo=0
dificultad=1

env = make_atari_env(
    "ALE/Freeway-v5",
    n_envs=1,
    seed=0,
    env_kwargs={"mode": modo, "difficulty": dificultad}
)

env = VecFrameStack(env, n_stack=4)  # apila 4 frames para dar percepción del movimiento

#Útil: Limitar el tiempo del episodio
#env = TimeLimit(env, max_episode_steps=512)

#Importante: Limitar el buffer por las limitaciones de Colab
#Útil: learning_Starts para ganar algo de experiencia (2 episodios)
model = QRDQN(
    "CnnPolicy",
    env,
    learning_rate=0.0001,
    buffer_size=50_000,         #original: 100_000
    learning_starts=1024,       #original: 100
    batch_size=64,
    gamma=0.99,
    train_freq=4,
    target_update_interval=5000, #original: 10000
    exploration_fraction=0.9,   #original: 0.1
    exploration_initial_eps=1.0,
    exploration_final_eps=0.2,  #original: 0.05
    policy_kwargs=dict(n_quantiles=101), #original: 51
    verbose=1)

#400 episodios de entrenamiento
model.learn(total_timesteps=204_800, log_interval=10)
model.save("qrdqn_freeway_1")


# Cargar el modelo entrenado
model = QRDQN.load("qrdqn_freeway_1")

# Crea un entorno para renderizar
env = gymnasium.make("ALE/Freeway-v5", render_mode="rgb_array", mode=modo, difficulty=dificultad)
env = renderlab.RenderFrame(env, "./output")

# Frame stack manual para el modelo (apilamos en gris 84x84)
frame_stack = deque(maxlen=4)

# Reset del entorno
obs, info = env.reset()

# Procesa una copia del frame sólo para el modelo
def preprocess(obs):
    gray = obs.mean(axis=2).astype(np.uint8)  # escala de grises
    resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return resized

# Llena el frame stack inicial
preprocessed = preprocess(obs)
for _ in range(4):
    frame_stack.append(preprocessed)

terminated = False
truncated = False
total_reward = 0

# Loop de simulación
while not (terminated or truncated):
    stacked_obs = np.stack(frame_stack, axis=0)  # (4, 84, 84) aquí se ajustan los frames
    action, _ = model.predict(stacked_obs, deterministic=True)

    # Avanza en el entorno real manteniendo la continuidad en el video
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    # Actualiza el stack sólo para el modelo
    preprocessed = preprocess(obs)
    frame_stack.append(preprocessed)

print("Recompensa obtenida en el episodio:", total_reward)
env.play()

Using cuda device
Wrapping the env in a VecTransposeImage.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0.1      |
|    exploration_rate | 0.978    |
| time/               |          |
|    episodes         | 10       |
|    fps              | 154      |
|    time_elapsed     | 32       |
|    total_timesteps  | 5092     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00813  |
|    n_updates        | 1016     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0.05     |
|    exploration_rate | 0.956    |
| time/               |          |
|    episodes         | 20       |
|    fps              | 150      |
|    time_elapsed     | 67       |
|    total_timesteps  | 10178    |
| train/              |          |
|    learning_rate    | 0.0001 

Moviepy - Done !
Moviepy - video ready temp-{start}.mp4


## Ejemplo Práctico

En el ejemplo anterior, se observa que se obtiene una recompensa cercana a 20. Esto es positivo, dado que entocnes el desempeño es entonces similar a cuando se jugaba con dificultad 0. Adicionalmente, hay una diferencia fundamental en la política observada en el video: el pollito ya no va siempre derecho, ahora frecuentemente se detiene, esperando a que pasen vehículos, y también retrocede para esquivarlos, algo que no se estaba observando con el DQN puro.

Repita el entrenamiento cambiando los parámetros:

*   modo 7 y dificultad 1.

También puede ajustar algunos de los siguientes parámetros del modelo y entrenamiento para intentar obtener mejores resultados:

*   <i>learning_rate</i>: La tasa de aprendizaje.
*   <i>batch_size</i>: Cantidad de experiencias almacenadas en el Experience Replay.
*   <i>exploration_fraction</i>: Porcentaje del entrenamiento en el cual el agente reduce progresivamente su tasa de exploración ($\epsilon$).
*   <i>exploration_final_eps</i>: La tasa de exploración ($\epsilon$) que mantendrá el agente al final.
*   <i>total_timesteps</i>: Equivalente al tiempo de entrenamiento.

In [ ]:
modo=7
dificultad=1

env = make_atari_env(
    "ALE/Freeway-v5",
    n_envs=1,
    seed=0,
    env_kwargs={"mode": modo, "difficulty": dificultad}
)

env = VecFrameStack(env, n_stack=4)  # apila 4 frames para dar percepción del movimiento

#Útil: Limitar el tiempo del episodio
#env = TimeLimit(env, max_episode_steps=512)

#Importante: Limitar el buffer por las limitaciones de Colab
#Útil: learning_Starts para ganar algo de experiencia (2 episodios)
model = QRDQN(
    "CnnPolicy",
    env,
    learning_rate=0.0001,
    buffer_size=50_000,         #original: 100_000
    learning_starts=1024,       #original: 100
    batch_size=64,
    gamma=0.99,
    train_freq=4,
    target_update_interval=5000, #original: 10000
    exploration_fraction=0.9,   #original: 0.1
    exploration_initial_eps=1.0,
    exploration_final_eps=0.2,  #original: 0.05
    policy_kwargs=dict(n_quantiles=101), #original: 51
    verbose=1)

#400 episodios de entrenamiento
model.learn(total_timesteps=204_800, log_interval=10)
model.save("qrdqn_freeway_2")

# Cargar el modelo entrenado
model = QRDQN.load("qrdqn_freeway_2")

# Crea un entorno para renderizar
env = gymnasium.make("ALE/Freeway-v5", render_mode="rgb_array", mode=modo, difficulty=dificultad)
env = renderlab.RenderFrame(env, "./output")

# Frame stack manual para el modelo (apilamos en gris 84x84)
frame_stack = deque(maxlen=4)

# Reset del entorno
obs, info = env.reset()

# Procesa una copia del frame sólo para el modelo
def preprocess(obs):
    gray = obs.mean(axis=2).astype(np.uint8)  # escala de grises
    resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return resized

# Llena el frame stack inicial
preprocessed = preprocess(obs)
for _ in range(4):
    frame_stack.append(preprocessed)

terminated = False
truncated = False
total_reward = 0

# Loop de simulación
while not (terminated or truncated):
    stacked_obs = np.stack(frame_stack, axis=0)  # (4, 84, 84) aquí se ajustan los frames
    action, _ = model.predict(stacked_obs, deterministic=True)

    # Avanza en el entorno real manteniendo la continuidad en el video
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    # Actualiza el stack sólo para el modelo
    preprocessed = preprocess(obs)
    frame_stack.append(preprocessed)

print("Recompensa obtenida en el episodio:", total_reward)
env.play()

Using cuda device
Wrapping the env in a VecTransposeImage.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.978    |
| time/               |          |
|    episodes         | 10       |
|    fps              | 149      |
|    time_elapsed     | 34       |
|    total_timesteps  | 5092     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00622  |
|    n_updates        | 1016     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.05e+03 |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.956    |
| time/               |          |
|    episodes         | 20       |
|    fps              | 149      |
|    time_elapsed     | 67       |
|    total_timesteps  | 10178    |
| train/              |          |
|    learning_rate    | 0.0001 

Moviepy - Done !
Moviepy - video ready temp-{start}.mp4


# Reflexiones Finales

Ahora que observó los resultados usando las diferentes versiones de DQN, puede reflexionar sobre las siguientes preguntas. Tenga en cuenta que, para algunos juegos de Atari, pueden ser necesarios millones de pasos y horas de entrenamiento, mientras que aquí únicamente se invirtieron algunos minutos. Teniendo en cuenta la limitante de tiempo:

*   ¿Es mejor usar DQN, o utilizar otra versión como QR-DQN o Doble DQN?

En general, es siempre conveniente implementar alguna de las mejoras que se han propuesto en lugar de DQN puro. Concretamente, Doble DQN es muy bueno para evitar el sesgo de maximización, lo cual en muchos problemas puede incurrir en malas políticas de comportamiento. Por otro lado, QR-DQN va más allá al modelar no solo el valor esperado de las recompensas, sino toda su distribución. Esto permite capturar mejor la incertidumbre y variabilidad del entorno, lo cual es crucial en juegos con alta aleatoriedad o múltiples trayectorias posibles. Aunque QR-DQN puede requerir más tiempo de entrenamiento, su capacidad para representar la distribución completa de retornos lo hace más robusto y estable, incluso en sesiones de entrenamiento cortas como las realizadas aquí.Adicionalmente, estrategias como Prioritized Experience Replay pueden ayudar mucho a discriminar las experiencias más significativas y hacer un uso más eficiente del tiempo disponible en una máquina.

*   ¿Cómo afecta la exploración en este caso?

La exploración en ejemplos como este donde el tiempo de exploración es pequeño, debe ser grande. Esto porque al inicio del entrenamiento el pollito probablemente no verá mucha recompensa después de muchos episodios. Si la exploración disminuye drásticamente antes de que el agente pueda identificar acciones acertadas, terminará explotando acciones que en realidad son malas, disminuyendo la recompensa al final del entrenamiento. Por ello, cuando hay pocos episodios disponibles, debe mantenerse una exploración significativa durante la mayoría del entrenamiento.

*   ¿Cómo afecta el tamaño de batch al entrenamiento?

Igualmente, como hay pocas experiencias disponibles, el tamaño de batch del Experience Replay ayuda a tener máyor cantidad de recursos en cuenta para realizar las actualizaciones. Un tamaño de batch más grande sería mejor, porque el agente tiene más información disponible y hace el entrenamiento más estable, pero esto consumiría mayor cantidad de memoria RAM, que no siempre está disponible.

*   ¿Por qué es importante usar redes neuronales convolucionales? ¿y por qué es importante usar escala de grises y apilar frames?

Siempre que se trabaja con imágenes se utilizan redes neuronales convolucionales debido a que la gran cantidad de pixeles representa mucha información, y esta arquitectura de red es la que permiteextraer información de forma eficiente en estos casos, llegando a correlaciones útiles que no serían posibles con redes simplemente conectadas. Adicionalmente, al usar la escala de grises se puede orientar al agente a buscar información más importante y agilizar el entrenamiento. Por ejemplo, en este problema el color de un carro no es reelevante, pero si es información que llega a la red, está demorará más en encontrar la relación adecuada. Sucede lo mismo con la aplicación de frames, ya que así se le da más datos del ambiente a la red en un único paso de tiempo, y puede diferenciar, por ejemplo, entre carros yendo a la izquierda o derecha para tomar decisiones más informadas.





# Referencias

[1] Mnih, V., Kavukcuoglu, K., Silver, D., Graves, A., Antonoglou, I., Wierstra, D., and Riedmiller, M. (2013). Playing atari with deep reinforcement learning. cite arxiv:1312.5602Comment: NIPS Deep Learning Workshop 2013.

[2] Hasselt, H. v., Guez, A., and Silver, D. (2016). Deep reinforcement
learning with double q-learning. In Proceedings of the Thirtieth AAAI Conference on Artificial Intelligence, AAAI'16, pages 2094-2100. AAAI Press.

[3] Schaul, T., Quan, J., Antonoglou, I., and Silver, D. (2015). Prioritized experience replay. cite arxiv:1511.05952Comment: Under review as a conference paper at ICLR 2016.

[4] Dabney, W., Rowland, M., Bellemare, M. G., and Munos, R. (2017). Distributional reinforcement learning with quantile regression. cite arxiv:1710.10044Comment: Published at AAAI 2018.

[5] Gym Documentation, Freeway. `https://gymnasium.farama.org/v0.28.1/environments/atari/freeway/`

[6] Stable Baselines3 Documentation, DQN. `https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html`